# Bài 11: Làm trơn và Kim tự tháp Gauss (Smoothing and Gaussian Pyramids)

Bài 10 đã giới thiệu làm mờ Gauss như một trong nhiều kernel tích chập. Ở đây ta xem xét *tại sao* làm mờ lại quan trọng, vượt ra ngoài việc chỉ "làm mềm" ảnh: nó là thành phần then chốt giúp việc giảm mẫu (downsampling) trở nên an toàn. Điều đó dẫn thẳng tới **kim tự tháp Gauss (Gaussian pyramid)** &mdash; một chồng các phiên bản ngày càng nhỏ hơn, mờ hơn của một ảnh, được dùng khắp nơi trong thị giác máy tính để phân tích đa tỉ lệ (multi-scale).

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['brick_wall.jpg', 'dog_looking.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tại sao giảm mẫu cần làm mờ trước (Why downsampling needs blurring first)
Thu nhỏ ảnh một cách ngây thơ bằng cách giữ lại mỗi điểm ảnh thứ $k$ ("giảm mẫu láng giềng gần nhất") có thể gây ra **răng cưa / chồng phổ (aliasing)**: các chi tiết tuần hoàn mịn dao động nhanh hơn mức mà khoảng cách điểm ảnh mới có thể biểu diễn sẽ bị gập lại thành một mẫu tần số thấp hoàn toàn khác, giả tạo.

Ta minh họa bằng ảnh chụp một bức tường gạch &mdash; các hàng gạch và đường vữa lặp lại chính là loại chi tiết mịn, đều đặn bị chồng phổ nặng nề.

In [ ]:
brick_wall = cv2.imread('../img/brick_wall.jpg', cv2.IMREAD_GRAYSCALE)

factor = 4
naive_downsample = brick_wall[::factor, ::factor]                                        # subsample directly
safe_downsample = cv2.GaussianBlur(brick_wall, (0, 0), sigmaX=factor / 2)[::factor, ::factor]  # blur first

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(brick_wall, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(naive_downsample, cmap='gray')
axes[1].set_title('Naive: subsample only\n(aliased, noisy-looking)')
axes[2].imshow(safe_downsample, cmap='gray')
axes[2].set_title('Blur, then subsample\n(correctly soft)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://www.publicdomainpictures.net/" target="_blank" rel="noopener">publicdomainpictures.net</a></p>

Phiên bản ngây thơ biến mẫu gạch đều đặn thành các đốm lấm tấm mịn giống nhiễu &mdash; một hiện tượng chồng phổ kinh điển (cũng chính là hiệu ứng khiến bánh xe ô tô trông như đang quay ngược trên máy quay). Phiên bản đã làm mờ vẫn nhận ra được là một bức tường gạch, vì các đường vữa đã được làm trơn *trước khi* lấy mẫu thưa, thay vì bị giữ lại hoặc bỏ đi ngẫu nhiên theo từng điểm ảnh.

## Chọn sigma và kích thước kernel (Choosing sigma and kernel size)
Làm mờ đúng cách trước khi giảm mẫu cần hai thứ để thực sự hiệu quả: `sigma` phù hợp, và một kernel *đủ lớn để biểu diễn sigma đó*. `sigma` của `cv2.GaussianBlur` quyết định trọng số hình chuông trải rộng đến đâu: sigma lớn hơn lấy trung bình trên vùng lân cận rộng hơn, loại bỏ chi tiết mịn hơn. Cụ thể, làm mờ một điểm ảnh bằng kernel Gauss 3x3 chẳng qua là một **trung bình có trọng số trên vùng lân cận 3x3 của nó**: nhân mỗi giá trị trong 9 điểm ảnh lân cận với trọng số kernel tương ứng rồi cộng lại. `cv2.getGaussianKernel(3, sigma)` cho ra các trọng số 1D; tích ngoài của chúng cho ra kernel 2D.

In [ ]:
rng = np.random.default_rng(0)
noise_img = rng.integers(0, 255, (20, 20)).astype(np.uint8)  # single-channel, for a simple example

sigma = 1.0
k1d = cv2.getGaussianKernel(3, sigma)
kernel2d = k1d @ k1d.T
print('3x3 Gaussian kernel (sigma=1.0):\n', np.round(kernel2d, 3))
print('weights sum to:', kernel2d.sum())

y, x = 10, 10
neighborhood = noise_img[y - 1:y + 2, x - 1:x + 2].astype(np.float64)
manual = (kernel2d * neighborhood).sum()

blurred_full = cv2.GaussianBlur(noise_img, (3, 3), sigmaX=sigma)

print(f'\n3x3 neighborhood around pixel ({x}, {y}):\n{neighborhood}')
print(f'\nweighted average (manual): {manual:.2f}')
print(f'cv2.GaussianBlur pixel:    {blurred_full[y, x]}')

Trung bình có trọng số tính thủ công khớp với đầu ra của `cv2.GaussianBlur` tại điểm ảnh đó (sai khác do làm tròn số nguyên) &mdash; mọi điểm ảnh đã làm mờ thực sự chỉ là tích vô hướng 9 số này, lặp lại tại mọi vị trí trong ảnh. Tuy nhiên, tăng sigma trong khi giữ kernel ở 3x3 hầu như không thay đổi gì, vì cửa sổ 3x3 chỉ có chỗ cho 3 phần tử &mdash; để thực sự thấy hiệu ứng của sigma rộng hơn, kernel cũng phải lớn lên.

In [ ]:
img = np.zeros((150, 150, 3), dtype=np.uint8)
cv2.circle(img, (75, 75), 55, (255, 120, 30), -1)
cv2.rectangle(img, (20, 20), (60, 60), (30, 200, 255), -1)

sigmas = [0, 1, 3, 8]
fig, axes = plt.subplots(1, len(sigmas), figsize=(12, 3.5))
for ax, s in zip(axes, sigmas):
    # ksize=(0, 0) tells OpenCV to pick a kernel size automatically, big enough for this sigma
    blurred = img if s == 0 else cv2.GaussianBlur(img, (0, 0), sigmaX=s)
    ax.imshow(blurred)
    ax.set_title(f'sigma = {s}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### Kích thước kernel phải theo kịp sigma (Kernel size has to keep up with sigma)
`ksize=(0, 0)` ở trên bảo OpenCV tự chọn kích thước kernel cho mỗi sigma. Điều gì xảy ra nếu thay vào đó kernel bị cố định quá nhỏ? `cv2.getGaussianKernel` luôn chuẩn hóa lại các trọng số để tổng bằng 1, bất kể số phần tử ít đến đâu &mdash; nên một kernel quá nhỏ không chỉ cắt mất hai đuôi của hàm Gauss, mà còn âm thầm biến đổi toàn bộ kernel thành thứ gần với trung bình hộp đơn thuần, vứt bỏ hoàn toàn hình dạng chuông.

In [ ]:
sigma = 5.0
k3 = cv2.getGaussianKernel(3, sigma).ravel()
print(f'a 3-tap kernel for sigma={sigma}, renormalized to sum to 1:', np.round(k3, 3))
print('(nearly identical to a plain 3-tap box average [0.33, 0.33, 0.33] -- the wide bell curve got squashed away)')

too_small = cv2.GaussianBlur(img, (3, 3), sigmaX=sigma)   # kernel far too small for this sigma
auto = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)         # OpenCV auto-sizes the kernel to fit sigma

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [img, too_small, auto],
                          ['Original', 'ksize=3\n(too small for sigma=5)', 'ksize=0\n(auto-sized for sigma=5)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'mean abs difference, too-small vs. correctly-sized kernel: {np.abs(too_small.astype(int) - auto.astype(int)).mean():.2f}')

Hai kết quả khác nhau thấy rõ &mdash; `ksize=3` hầu như không làm mờ ảnh chút nào, dù yêu cầu `sigma=5`, vì một cửa sổ 3 phần tử đơn giản là không thể biểu diễn một đường cong chuông rộng đến thế. Đây chính là lý do đoạn code kim tự tháp bên dưới (và cách khắc phục chồng phổ ở trên) truyền `ksize=(0, 0)`: để OpenCV chọn kernel đủ rộng cho sigma được yêu cầu, thay vì mạo hiểm với một kernel nhỏ quá mức mà không hề hay biết.

## Xây dựng kim tự tháp Gauss (Building a Gaussian pyramid)
**Kim tự tháp Gauss** lặp đi lặp lại thao tác "làm mờ, rồi giảm mẫu 2 lần", tạo ra một chồng ảnh mà mỗi ảnh có chiều rộng và chiều cao bằng một nửa ảnh trước đó. Mỗi tầng là một góc nhìn thô hơn, đã được khử răng cưa đúng cách, của cùng một cảnh &mdash; không chỉ là một vùng cắt nhỏ hơn.

In [ ]:
def gaussian_pyramid(image, num_levels, sigma=1.0):
    pyramid = [image]
    current = image
    for _ in range(num_levels - 1):
        blurred = cv2.GaussianBlur(current, (0, 0), sigmaX=sigma)
        current = blurred[::2, ::2]
        pyramid.append(current)
    return pyramid

photo = np.zeros((256, 256, 3), dtype=np.uint8)
photo[:] = (40, 40, 40)
cv2.circle(photo, (128, 128), 90, (255, 120, 30), -1)
cv2.rectangle(photo, (30, 30), (110, 110), (30, 200, 255), -1)

pyramid = gaussian_pyramid(photo, num_levels=5)

fig, axes = plt.subplots(1, len(pyramid), figsize=(13, 3))
for ax, level in zip(axes, pyramid):
    ax.imshow(level)
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Dưới đây là cùng quá trình đó dưới dạng ảnh động trên ảnh chụp thật một chú chó. Cột giữa hiển thị các tầng đã hoàn thành xếp chồng thành kim tự tháp (nhìn 3D từ bên cạnh). Cột bên phải hiển thị cùng các tầng đó được đổi về cùng một kích thước, để dễ thấy chi tiết mất dần.

<img src="../animations/pyramid_demo.gif" alt="Ảnh động xây dựng kim tự tháp Gauss trên ảnh chụp thật một chú chó: làm mờ rồi giảm mẫu, lặp lại, với các tầng thu nhỏ dần được hiển thị thành chồng kim tự tháp bậc thang 3D và đồng thời được phóng về cùng kích thước để thấy rõ sự mất chi tiết" style="display: block; margin: 0 auto; max-width: 820px;">

<p class="image-attribution">Ảnh của <a href="https://unsplash.com/@laulaco?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Laula Co</a> trên <a href="https://unsplash.com/photos/long-coat-tan-puppy-sICtHYZzt_s?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Unsplash</a></p>

### So sánh với hàm có sẵn `cv2.pyrDown` của OpenCV (Comparison to OpenCV's built-in `cv2.pyrDown`)
OpenCV cung cấp `cv2.pyrDown`, thực hiện cùng ý tưởng làm-mờ-rồi-giảm-mẫu nhưng dùng một kernel nhị thức 5 phần tử cố định, được thiết kế cẩn thận (xấp xỉ hàm Gauss) thay vì một `sigma` tùy ý. Kích thước đầu ra khớp chính xác với của ta; giá trị điểm ảnh gần giống nhưng không hoàn toàn trùng, vì các kernel hơi khác nhau.

In [ ]:
cv_pyramid = [photo]
current = photo
for _ in range(4):
    current = cv2.pyrDown(current)
    cv_pyramid.append(current)

for ours, cvs in zip(pyramid, cv_pyramid):
    assert ours.shape == cvs.shape
    diff = np.abs(ours.astype(int) - cvs.astype(int))
    print(f'{ours.shape[1]:>4}x{ours.shape[0]:<4}  mean abs diff = {diff.mean():.2f}')

## Kim tự tháp làm mất thông tin (Pyramids lose information)
Giảm mẫu là không thể đảo ngược: tăng mẫu (upsampling) một tầng thấp của kim tự tháp trở về kích thước gốc (`cv2.pyrUp`) không thể khôi phục chi tiết đã bị làm mờ/lấy mẫu thưa loại bỏ. Khoảng chênh lệch giữa một tầng thô đã tăng mẫu và ảnh gốc chính là thứ mà **kim tự tháp Laplace (Laplacian pyramid)** nắm bắt ở mỗi tầng &mdash; chủ đề của một bài sau &mdash; nhưng ngay bây giờ ta đã có thể thấy trực tiếp sự mất thông tin.

In [ ]:
level1 = pyramid[1]                              # 128x128, one pyrDown from the original
reconstructed = cv2.pyrUp(level1)                 # back up to 256x256

diff = cv2.absdiff(photo, reconstructed)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [photo, reconstructed, diff],
                          ['Original', 'pyrDown then pyrUp', 'Difference (lost detail)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'mean absolute reconstruction error = {diff.astype(np.float64).mean():.2f} gray levels')

Các cạnh sắc nét của hình tròn và hình vuông trở lại mềm và trông như bị lệch &mdash; chi tiết mịn đã bị loại bỏ vĩnh viễn khi ảnh bị làm mờ và lấy mẫu thưa, và `pyrUp` (nội suy) chỉ có thể đoán, chứ không thể khôi phục nó.

### Bài tập (Exercises)
1. Lặp lại thí nghiệm bức tường gạch với `factor = 2` thay vì `4`. Lấy mẫu thưa ngây thơ có còn cho thấy chồng phổ rõ rệt không? Tại sao hệ số giảm mẫu nhỏ hơn lại có thể ít chồng phổ hơn?
2. Kim tự tháp Gauss được dùng cho tìm kiếm từ-thô-đến-mịn (coarse-to-fine) (ví dụ nhanh chóng tìm vị trí gần đúng của đối tượng ở một tầng kim tự tháp nhỏ, rồi tinh chỉnh ở các tầng lớn hơn). Tại sao bỏ qua bước làm mờ và chỉ lấy mẫu thưa trực tiếp lại phá hỏng chiến lược này?
3. Xây dựng một kim tự tháp 6 tầng của một ảnh giống ảnh chụp thật và đo xem cần bao nhiêu tầng thì ảnh trở nên quá nhỏ để nhận ra bất kỳ hình dạng nào. Với một bức ảnh 1920x1080 chẳng hạn, điều đó tương ứng với độ phân giải thực tế bao nhiêu?